# Modeling Human Activity States Using Hidden Markov Models

## Background and Motivation

In real-world systems, from wearable health monitors to smart home sensors, continuous data streams such as accelerometer and gyroscope signals reveal valuable information about human activity. However, the true activity state (e.g., walking or standing) is often **hidden** behind noisy measurements. Hidden Markov Models (HMMs) are a natural fit for this problem: the activities are the hidden states, and the sensor readings are the noisy observations.

This notebook implements a Gaussian HMM **from scratch using NumPy** to infer four human activities (Standing, Walking, Jumping, Still) from smartphone accelerometer and gyroscope data collected using the **Sensor Logger** app.

## Data Collection Summary

Data was collected by two group members using different smartphones. Each member recorded 5-10 second clips of each activity, yielding the following dataset:

| Phone | Device | Platform | Sampling Rate | Date Collected |
|-------|--------|----------|---------------|----------------|
| Phone 1 | STK-L22 | Android | 20ms (50 Hz) | 2026-03-04 |
| Phone 2 | Infinix X6855 | Android | 20ms (50 Hz) | 2026-02-28 |

Both phones use the **same 20ms sampling rate**, so no resampling or harmonization is needed. Each recording was saved via Sensor Logger as a zip file containing `Accelerometer.csv` and `Gyroscope.csv`, which were then extracted, merged by nearest timestamp, and saved as clean labelled CSV files.

## Notebook Outline

1. **Data Loading** - Load 57 cleaned CSV files and summarize the dataset
2. **Raw Signal Visualization** - Plot sample sensor signals per activity
3. **Windowing** - Segment continuous signals into fixed-size windows
4. **Feature Extraction** - Compute time-domain and frequency-domain features
5. **Train/Test Split** - Hold out files for unseen evaluation
6. **HMM Definition** - Define model components (states, observations, A, B, pi)
7. **Baum-Welch Training** - Train the HMM using the EM algorithm
8. **Visualization** - Transition matrix heatmap, emission distributions, decoded sequences
9. **Evaluation on Unseen Data** - Test on combined multi-activity sequences
10. **Analysis and Reflection** - Discuss results, limitations, and improvements

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.fft import fft, fftfreq
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.preprocessing import StandardScaler
import os
import glob
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print("Imports loaded successfully.")

## 1. Data Loading

We load all 57 cleaned CSV files from the `data/` directory. The files are organized into four subfolders by activity (`Standing/`, `Walking/`, `Jumping/`, `Still/`), and each file is named using the convention `activity_phoneN_NN.csv` to identify the source phone and recording number.

Each CSV contains 7 columns:
- `seconds_elapsed` - timestamp relative to the start of the recording
- `accel_x, accel_y, accel_z` - accelerometer readings (m/sÂ²)
- `gyro_x, gyro_y, gyro_z` - gyroscope readings (rad/s)

After loading, we print a summary table showing the number of files, phone distribution, and total recording duration per activity. This confirms that we meet the requirement of **50+ files** and **at least 1 minute 30 seconds** per activity.

In [ ]:
DATA_DIR = "data"
ACTIVITIES = ["Standing", "Walking", "Jumping", "Still"]
SAMPLING_RATE = 50  # Hz (20ms interval)

# Load all files
all_data = []  # list of (df, activity, phone, filename)

for activity in ACTIVITIES:
    activity_dir = os.path.join(DATA_DIR, activity)
    csv_files = sorted(glob.glob(os.path.join(activity_dir, "*.csv")))
    for fpath in csv_files:
        fname = os.path.basename(fpath)
        phone = "phone1" if "phone1" in fname else "phone2"
        df = pd.read_csv(fpath)
        all_data.append((df, activity, phone, fname))

print(f"Total files loaded: {len(all_data)}\n")

# Summary table
summary = []
for activity in ACTIVITIES:
    files = [(df, phone) for df, act, phone, _ in all_data if act == activity]
    n_files = len(files)
    total_dur = sum(df["seconds_elapsed"].max() - df["seconds_elapsed"].min() for df, _ in files)
    n_p1 = sum(1 for _, p in files if p == "phone1")
    n_p2 = sum(1 for _, p in files if p == "phone2")
    summary.append([activity, n_files, n_p1, n_p2, f"{total_dur:.1f}s"])

summary_df = pd.DataFrame(summary, columns=["Activity", "Files", "Phone1", "Phone2", "Total Duration"])
print(summary_df.to_string(index=False))

**Data loading results:** We successfully loaded **57 CSV files** across the 4 activities. Phone 1 (STK-L22) contributed 7 files per activity (28 total), and Phone 2 (Infinix X6855) contributed 7-8 files per activity (29 total). The total recording duration per activity ranges from **112.8s to 132.2s**, all exceeding the minimum requirement of 1 minute 30 seconds. The dataset is well-balanced across activities and phones.

Next, we visualize the raw sensor signals to understand the characteristic patterns of each activity.

## 2. Raw Signal Visualization

Before any processing, we visualize the raw accelerometer and gyroscope signals for one sample recording from each activity. This serves two purposes:

1. **Verify data quality** - confirm that the recordings are clean and the sensor readings look reasonable for each activity.
2. **Build intuition** - understand how different activities produce distinct signal patterns:
   - **Standing**: relatively flat signals with minor body sway
   - **Walking**: rhythmic oscillations from the step pattern
   - **Jumping**: large, high-frequency spikes in acceleration
   - **Still**: near-constant values with minimal noise (phone on a flat surface)

The plot below shows a 4x2 grid: one row per activity, with accelerometer (left) and gyroscope (right) signals on all three axes (x, y, z).

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(14, 12))
fig.suptitle("Raw Sensor Signals by Activity", fontsize=14, y=1.01)

for i, activity in enumerate(ACTIVITIES):
    # Pick first file for this activity
    df = [d for d, act, _, _ in all_data if act == activity][0]
    t = df["seconds_elapsed"]
    
    # Accelerometer
    ax = axes[i, 0]
    ax.plot(t, df["accel_x"], label="x", alpha=0.8)
    ax.plot(t, df["accel_y"], label="y", alpha=0.8)
    ax.plot(t, df["accel_z"], label="z", alpha=0.8)
    ax.set_ylabel(f"{activity}\nAccel (m/s²)")
    ax.legend(loc="upper right", fontsize=7)
    if i == 0:
        ax.set_title("Accelerometer")
    
    # Gyroscope
    ax = axes[i, 1]
    ax.plot(t, df["gyro_x"], label="x", alpha=0.8)
    ax.plot(t, df["gyro_y"], label="y", alpha=0.8)
    ax.plot(t, df["gyro_z"], label="z", alpha=0.8)
    ax.legend(loc="upper right", fontsize=7)
    if i == 0:
        ax.set_title("Gyroscope")

axes[-1, 0].set_xlabel("Time (s)")
axes[-1, 1].set_xlabel("Time (s)")
plt.tight_layout()
plt.show()

**Observations from the raw signals:**

- **Standing** shows relatively stable accelerometer readings with small fluctuations caused by natural body sway. The gyroscope shows minor rotational noise.
- **Walking** exhibits clear periodic oscillations in the accelerometer (especially the y and z axes) corresponding to the gait cycle (~2 steps/second). The gyroscope captures the rhythmic hip/leg rotation.
- **Jumping** produces the largest acceleration spikes across all axes, with sharp peaks at each landing. The gyroscope also shows high variability from the body rotation during jumps.
- **Still** is the flattest - nearly constant accelerometer values (dominated by gravity on the z-axis) and near-zero gyroscope readings, since the phone is resting on a flat surface.

These visual differences confirm that the four activities have distinct signal signatures, which gives us confidence that an HMM should be able to distinguish them. Next, we segment these continuous signals into fixed-size **windows** for feature extraction.

## 3. Windowing

Before extracting features, we divide each continuous recording into fixed-size **windows** (segments). This is necessary because:
- The HMM operates on a sequence of discrete observations, not raw sample-by-sample data.
- Features computed over a window (e.g., mean, variance, FFT) capture the statistical properties of the signal within that time interval.

### Window Parameters

| Parameter | Value | Rationale |
|-----------|-------|-----------|
| **Sampling rate** | 50 Hz (20ms per sample) | Both phones record at this rate |
| **Window size** | 50 samples = **1 second** | Captures a full gait cycle for walking (~0.5-1.0s) and at least one jump cycle. Long enough for meaningful FFT resolution (1 Hz) while short enough to yield many windows per 5-10s recording. |
| **Overlap** | 50% (step = 25 samples) | Increases the number of training windows without losing temporal continuity between adjacent windows. |

### Why 1 second?

At 50 Hz, a 1-second window contains 50 samples. This is the ideal trade-off:
- **Too short** (e.g., 0.25s = 12 samples): not enough data for stable statistics or FFT.
- **Too long** (e.g., 3s = 150 samples): fewer windows per file, and the window might span activity transitions in combined sequences.
- **1 second**: captures full movement cycles, yields ~8-16 windows per file, and provides 1 Hz FFT resolution.

In [ ]:
WINDOW_SIZE = 50   # 1 second at 50 Hz
STEP_SIZE = 25     # 50% overlap
SENSOR_COLS = ["accel_x", "accel_y", "accel_z", "gyro_x", "gyro_y", "gyro_z"]

def segment_windows(df, window_size=WINDOW_SIZE, step_size=STEP_SIZE):
    """Segment a dataframe into fixed-size windows."""
    values = df[SENSOR_COLS].values
    windows = []
    for start in range(0, len(values) - window_size + 1, step_size):
        windows.append(values[start:start + window_size])
    return windows

# Segment all files
all_windows = []   # list of (window_array, activity_label)
for df, activity, phone, fname in all_data:
    windows = segment_windows(df)
    for w in windows:
        all_windows.append((w, activity))

print(f"Total windows extracted: {len(all_windows)}")
for act in ACTIVITIES:
    n = sum(1 for _, a in all_windows if a == act)
    print(f"  {act}: {n} windows")

**Windowing results:** We extracted **663 windows** in total across all 57 files and 4 activities. The distribution is roughly balanced (157-179 per activity), which is important for training an unbiased model. Each window is a (50, 6) matrix of raw sensor values. Next, we extract meaningful features from each window to form the observation vectors for the HMM.

## 4. Feature Extraction

We extract both **time-domain** and **frequency-domain** features from each window. These features form the observation vectors for the HMM.

### Time-Domain Features (22 features)

| Feature | Count | Justification |
|:---|:---:|:---|
| Mean (per axis) | 6 | Baseline signal level; Standing has a distinct z-accel value from gravity |
| Standard Deviation (per axis) | 6 | Measures movement intensity; Jumping has high std, Still has near-zero |
| RMS (per axis) | 6 | Overall signal energy; higher for dynamic activities like Walking and Jumping |
| Signal Magnitude Area (SMA) | 1 | Total body acceleration summed across all accelerometer axes |
| Inter-axis Correlation (xy, xz, yz) | 3 | Walking shows correlated axes from the rhythmic, coordinated motion pattern |

### Frequency-Domain Features (6 features)

| Feature | Count | Justification |
|:---|:---:|:---|
| Dominant Frequency (per accel axis) | 3 | Walking peaks at ~2 Hz from regular steps; Still has no dominant peak |
| Spectral Energy (per accel axis) | 3 | Separates high-energy Jumping from low-energy Still |

**Total: 28 features per window.**

The frequency-domain features are computed using the **Fast Fourier Transform (FFT)** on each accelerometer axis.

In [ ]:
# Define feature extraction function
# Extract 28 features per window (22 time-domain + 6 frequency-domain)
# Build feature matrix X_raw and label arrays

### 4.1 Feature Normalization

All 28 features are normalized using **Z-score standardization** (zero mean, unit variance). This is necessary because:

- Features have different units (m/sÂ² for accelerometer, rad/s for gyroscope, Hz for frequency, dimensionless for correlations)
- Gaussian emission distributions in the HMM assume comparable feature scales
- Without normalization, features with larger magnitudes would dominate the distance calculations

Z-score was chosen over min-max normalization because it preserves the shape of the feature distributions and is robust to outliers.

In [ ]:
# Apply Z-score normalization using StandardScaler
# Verify mean ~0 and std ~1 after normalization

## 5. Train / Test Split

We split the data **by file** (not by individual windows) to ensure that the test set contains genuinely unseen recordings. If we split by window, adjacent windows from the same file could end up in both train and test sets, inflating accuracy due to data leakage.

**Strategy:** Hold out the **last 2 files per activity** (8 files total) for testing. These are all from Phone 2 (the partner's recordings from a different date and device), making the test a cross-session evaluation. The remaining 49 files are used for training.

In [ ]:
# Build per-file window indices so we can split by file
file_window_ranges = []  # (start_idx, end_idx, activity, fname)
idx = 0
for df, activity, phone, fname in all_data:
    n_windows = len(segment_windows(df))
    file_window_ranges.append((idx, idx + n_windows, activity, fname))
    idx += n_windows

# Hold out last 2 files per activity for testing
test_indices = []
train_indices = []

for activity in ACTIVITIES:
    activity_files = [(s, e, a, f) for s, e, a, f in file_window_ranges if a == activity]
    # Last 2 files -> test
    for s, e, a, f in activity_files[:-2]:
        train_indices.extend(range(s, e))
    for s, e, a, f in activity_files[-2:]:
        test_indices.extend(range(s, e))

train_indices = np.array(train_indices)
test_indices = np.array(test_indices)

X_train, y_train = X[train_indices], y[train_indices]
X_test, y_test = X[test_indices], y[test_indices]

print(f"Training: {len(X_train)} windows")
print(f"Testing:  {len(X_test)} windows")
for i, act in enumerate(ACTIVITIES):
    print(f"  {act} -> train: {np.sum(y_train == i)}, test: {np.sum(y_test == i)}")

**Split results:** 588 windows for training and 75 windows for testing, with a roughly balanced distribution across all 4 activities. The test windows come from 8 held-out files recorded on Phone 2 (Infinix X6855) on 2026-02-28, which were not seen during training.

Now we define the HMM structure and implement the algorithms.

## 6. HMM Model Definition

Our Hidden Markov Model is defined by the following five components:

| Component | Symbol | Description |
|:---|:---:|:---|
| Hidden States | Z | 4 activities: Standing, Walking, Jumping, Still |
| Observations | X | 28-dimensional feature vectors from windowed sensor data |
| Transition Matrix | A (4x4) | Probability of transitioning between activities |
| Emission Parameters | B = {mu_k, sigma_k} | Gaussian mean and diagonal covariance per state |
| Initial Probabilities | pi (4x1) | Probability of starting in each activity |

We use **diagonal covariance** matrices for computational stability and to avoid singularity issues with limited training data. All computations are performed in **log-space** to prevent numerical underflow in the Forward, Backward, and Viterbi algorithms.

The model is implemented from scratch as a `GaussianHMM` class with the following methods:
- `initialize_from_labels()` - set initial parameters from ground-truth labels
- `_forward()` / `_backward()` - Forward and Backward algorithms in log-space
- `fit()` - Baum-Welch (EM) training
- `viterbi()` - Viterbi decoding for most-likely state sequence
- `predict()` - wrapper for Viterbi decoding

In [ ]:
# Define GaussianHMM class with:
# - __init__: initialize parameters (pi, A, means, covars)
# - initialize_from_labels: set params from ground-truth for faster convergence
# - _log_gaussian_pdf: log emission probability under diagonal Gaussian
# - _compute_log_emission: emission matrix for all observations
# - _forward: Forward algorithm in log-space
# - _backward: Backward algorithm in log-space
# - _logsumexp: numerically stable log-sum-exp
# - fit: Baum-Welch (EM) training with convergence check
# - viterbi: Viterbi decoding (dynamic programming in log-space)
# - predict: wrapper for viterbi

## 7. Training with Baum-Welch (EM Algorithm)

We now train the HMM using the **Baum-Welch algorithm**, which is an Expectation-Maximization (EM) procedure:

1. **E-step (Expectation):** Run the Forward and Backward algorithms on each training sequence to compute:
   - **Gamma** (gamma_t(k)): the posterior probability of being in state k at time t, given the full observation sequence.
   - **Xi** (xi_t(i,j)): the posterior probability of transitioning from state i to state j at time t.

2. **M-step (Maximization):** Update all model parameters (pi, A, means, covariances) using the soft state assignments from the E-step.

3. **Convergence check:** After each iteration, compute the total **log-likelihood** of the training data. Stop when the change in log-likelihood falls below **epsilon = 1e-4**, indicating the parameters have stabilized.

We organize the training data as **49 separate sequences** (one per training file), since each file is an independent recording. This is important because the HMM's initial state probabilities (pi) apply at the start of each sequence, not just once globally.

First, we build the training sequences and initialize the model parameters from the labelled data.

In [ ]:
# Build training sequences (one per file)
train_sequences = []
train_seq_labels = []

for df, activity, phone, fname in all_data:
    windows = segment_windows(df)
    if len(windows) == 0:
        continue
    feats = np.array([extract_features(w) for w in windows])
    feats_scaled = scaler.transform(feats)
    label_idx = label_to_idx[activity]
    
    # Check if this file's windows are in the training set
    file_range = None
    for s, e, a, f in file_window_ranges:
        if f == fname:
            file_range = (s, e)
            break
    if file_range and file_range[0] in train_indices:
        train_sequences.append(feats_scaled)
        train_seq_labels.append(label_idx)

print(f"Number of training sequences: {len(train_sequences)}")
print(f"Sequence lengths: {[len(s) for s in train_sequences]}")

**Training sequences:** We have **49 training sequences** (one per training file), with lengths ranging from 8 to 16 windows each. Each sequence is a single-activity recording, which means the model will learn strong self-transition probabilities.

Now we initialize the model from the labelled data and run Baum-Welch training.

In [ ]:
# Initialize and train the HMM
N_STATES = 4
N_FEATURES = X_train.shape[1]

model = GaussianHMM(N_STATES, N_FEATURES)
model.initialize_from_labels(X_train, y_train)

print(f"Training HMM with {N_STATES} states and {N_FEATURES} features...")
print(f"Convergence criterion: |delta log-likelihood| < 1e-4\n")

log_likelihoods = model.fit(train_sequences, max_iter=150, tol=1e-4, verbose=True)

print(f"\nFinal log-likelihood: {log_likelihoods[-1]:.4f}")
print(f"Total iterations: {len(log_likelihoods)}")

**Baum-Welch training output:** The model converged after the log-likelihood stabilized. The convergence plot below visualizes this - we expect a monotonically increasing curve that flattens at convergence.

In [ ]:
# Plot convergence
plt.figure(figsize=(8, 4))
plt.plot(log_likelihoods, marker='o', markersize=3)
plt.xlabel("Iteration")
plt.ylabel("Total Log-Likelihood")
plt.title("Baum-Welch Convergence")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Training results:** The Baum-Welch algorithm converged in **6 iterations**, with the total log-likelihood increasing from ~4,509 to ~7,682. The rapid convergence is due to our label-based initialization - the initial parameter estimates were already close to optimal. The convergence plot above confirms that the log-likelihood monotonically increases and plateaus, which is the expected behavior of a correctly implemented EM algorithm.

With the model trained, we now visualize the learned parameters to understand what the model has captured about each activity.

## 8. Transition Matrix Heatmap

The transition matrix **A** captures the probability of moving from one activity state to another between consecutive time windows. We visualize it as a heatmap to understand the learned activity dynamics.

**Expected behavior:**
- High diagonal values (close to 1.0) indicate that activities are persistent - a person who is walking is very likely to still be walking in the next 1-second window
- Small off-diagonal values allow the model to handle genuine activity transitions while resisting spurious state changes from noise

In [ ]:
# Plot transition matrix A as a heatmap using seaborn

## 9. Emission Probability Visualization

Since our HMM uses **Gaussian emissions**, each state k is characterized by a mean vector (mu_k) and a variance vector (sigma_k). We visualize these for a subset of 9 key features to understand what signal patterns the model associates with each activity.

- **Left heatmap (Emission Means):** Shows the average feature value for each state. For example, Jumping should have high `std_accel` and `energy` means, while Still should have low values.
- **Right heatmap (Emission Variances):** Shows how spread out the observations are for each state. Higher variance means more variability in that feature for that activity.

In [ ]:
# Plot emission means for selected features
key_features = ["mean_accel_x", "mean_accel_y", "mean_accel_z",
                "std_accel_x", "std_accel_y", "std_accel_z",
                "sma", "dom_freq_ax", "energy_ax"]
key_indices = [FEATURE_NAMES.index(f) for f in key_features]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Emission means
emission_means_df = pd.DataFrame(
    model.means[:, key_indices],
    index=ACTIVITIES,
    columns=key_features
)
sns.heatmap(emission_means_df, annot=True, fmt=".2f", cmap="coolwarm", ax=axes[0])
axes[0].set_title("Emission Means (B) - Key Features")

# Emission variances
emission_vars_df = pd.DataFrame(
    model.covars[:, key_indices],
    index=ACTIVITIES,
    columns=key_features
)
sns.heatmap(emission_vars_df, annot=True, fmt=".2f", cmap="Purples", ax=axes[1])
axes[1].set_title("Emission Variances (B) - Key Features")

plt.tight_layout()
plt.show()

**Interpretation of emission parameters:** The emission means show clear separation between activity states:
- **Jumping** has the highest values for `std_accel`, `sma`, and `energy` features - consistent with the high-energy, high-variance nature of jumping.
- **Still** has the lowest values across most features, reflecting the near-zero signal variability when the phone is stationary.
- **Standing** and **Still** have similar means for some features (e.g., `mean_accel`), which explains why these two are the hardest to distinguish.
- **Walking** sits between the static and dynamic extremes, with moderate std and energy values.

The variance heatmap shows that **Jumping** has the highest emission variance, meaning the model allows more spread in its observations - this makes sense because jumping produces more variable signals than other activities.

Next, we use the **Viterbi algorithm** to decode the most likely activity sequence from the training data, verifying that the model can recover the correct states.

## 10. Viterbi Decoding on Training Data

The **Viterbi algorithm** finds the single most likely sequence of hidden states given the observations. It uses dynamic programming in log-space to compute the maximum probability path through the state trellis, then backtracks to recover the optimal state sequence.

We first run Viterbi on the training sequences to verify that the model has learned the correct state assignments. For each activity, we show one training sequence with the true state (black) overlaid with the predicted state (colored).

In [ ]:
# Run Viterbi on training sequences
# Plot true vs predicted state sequences (4x1 subplot grid)

## 11. Evaluation on Unseen Data

We test the trained model on **combined multi-activity sequences** built from held-out files
(the last 2 files per activity, not used during training). Instead of feeding single-activity
files individually, we concatenate the **raw sensor data** from one file per activity into a
single continuous stream, then apply windowing over this combined signal:

```
[Standing raw data] Ã¢â€ â€™ [Walking raw data] Ã¢â€ â€™ [Jumping raw data] Ã¢â€ â€™ [Still raw data]
```

This creates a realistic challenge: windows near activity **transition boundaries** contain
mixed signals from two activities, forcing the model to handle ambiguous observations.
This tests the HMM's core strength - modeling temporal state transitions and recovering
from boundary ambiguity using transition probabilities.

In [ ]:
# Collect test file raw DataFrames grouped by activity
test_files_raw = {i: [] for i in range(N_STATES)}

for df, activity, phone, fname in all_data:
    label_idx = label_to_idx[activity]
    file_range = None
    for s, e, a, f in file_window_ranges:
        if f == fname:
            file_range = (s, e)
            break
    if file_range and file_range[0] in test_indices:
        test_files_raw[label_idx].append((fname, df))

print("Test files available per activity:")
for i, act in enumerate(ACTIVITIES):
    print(f"  {act}: {[f for f, _ in test_files_raw[i]]}")

# Build 2 combined test sequences from RAW data, then window + extract features
combined_sequences = []
combined_labels = []
combined_boundaries = []  # window-level boundaries for visualization

for seq_num in range(2):
    # Concatenate raw sensor data
    raw_parts = []
    raw_label_per_sample = []
    sample_boundaries = [0]
    
    for act_idx in range(N_STATES):
        fname, df = test_files_raw[act_idx][seq_num]
        raw_vals = df[SENSOR_COLS].values
        raw_parts.append(raw_vals)
        raw_label_per_sample.append(np.full(len(raw_vals), act_idx))
        sample_boundaries.append(sample_boundaries[-1] + len(raw_vals))
    
    combined_raw = np.vstack(raw_parts)
    combined_sample_labels = np.concatenate(raw_label_per_sample)
    
    # Window over the combined raw signal
    windows = []
    window_labels = []
    window_boundaries = [0]
    
    for start in range(0, len(combined_raw) - WINDOW_SIZE + 1, STEP_SIZE):
        window = combined_raw[start:start + WINDOW_SIZE]
        windows.append(window)
        # Label = majority activity in the window
        window_sample_labels = combined_sample_labels[start:start + WINDOW_SIZE]
        majority_label = np.bincount(window_sample_labels.astype(int)).argmax()
        window_labels.append(majority_label)
    
    # Extract features and scale
    feats = np.array([extract_features(w) for w in windows])
    feats_scaled = scaler.transform(feats)
    labels_arr = np.array(window_labels)
    
    # Find window-level boundaries (where majority label changes)
    boundaries = [0]
    for w_idx in range(1, len(labels_arr)):
        if labels_arr[w_idx] != labels_arr[w_idx - 1]:
            boundaries.append(w_idx)
    boundaries.append(len(labels_arr))
    
    combined_sequences.append(feats_scaled)
    combined_labels.append(labels_arr)
    combined_boundaries.append(boundaries)
    
    print(f"\nCombined Test Sequence {seq_num + 1}: {len(feats_scaled)} windows")
    for act_idx in range(N_STATES):
        n = np.sum(labels_arr == act_idx)
        print(f"  {ACTIVITIES[act_idx]}: {n} windows")

**Test sequence construction:** We built **2 combined test sequences** by concatenating raw sensor data from one held-out file per activity in the order Standing Ã¢â€ â€™ Walking Ã¢â€ â€™ Jumping Ã¢â€ â€™ Still. Sequence 1 has 41 windows and Sequence 2 has 42 windows. Because windowing is applied over the combined raw data, windows near activity boundaries contain mixed signals from two activities, creating realistic transition challenges for the model.

Now we run Viterbi decoding on these combined sequences and evaluate the predictions.

In [ ]:
# Run Viterbi on combined test sequences
all_true = []
all_pred = []

for i, (seq, true_labels) in enumerate(zip(combined_sequences, combined_labels)):
    pred_path = model.predict(seq)
    all_true.extend(true_labels)
    all_pred.extend(pred_path)
    
    acc = np.mean(pred_path == true_labels)
    print(f"Combined Sequence {i+1}: accuracy = {acc:.2%}")

all_true = np.array(all_true)
all_pred = np.array(all_pred)

# Confusion Matrix
cm = confusion_matrix(all_true, all_pred, labels=range(N_STATES))

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=ACTIVITIES, yticklabels=ACTIVITIES)
plt.title("Confusion Matrix (Combined Multi-Activity Test Sequences)")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
plt.show()

**Confusion matrix interpretation:** The confusion matrix above shows the number of windows classified into each activity. The diagonal cells represent correct predictions, while off-diagonal cells represent misclassifications. A perfect model would have all counts on the diagonal and zeros elsewhere.

In our results, the matrix is almost entirely diagonal, with only a small number of off-diagonal errors. The misclassification occurs between **Standing** and **Still** - the two most similar activities - specifically at the transition boundary where the windowed signal contains mixed data from both activities.

Next, we compute the standard classification metrics (sensitivity, specificity, and accuracy) per activity from this confusion matrix.

In [ ]:
# Compute Sensitivity, Specificity, and Accuracy per activity
print("\nEvaluation Results on Unseen Data")
print("=" * 70)
print(f"{'Activity':<15} {'Samples':>8} {'Sensitivity':>12} {'Specificity':>12} {'Accuracy':>10}")
print("-" * 70)

overall_correct = 0
overall_total = 0

for k in range(N_STATES):
    tp = cm[k, k]
    fn = cm[k, :].sum() - tp
    fp = cm[:, k].sum() - tp
    tn = cm.sum() - tp - fn - fp
    
    n_samples = cm[k, :].sum()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    accuracy = (tp + tn) / cm.sum() if cm.sum() > 0 else 0.0
    
    print(f"{ACTIVITIES[k]:<15} {n_samples:>8d} {sensitivity:>11.2%} {specificity:>11.2%} {accuracy:>9.2%}")
    overall_correct += tp
    overall_total += n_samples

print("-" * 70)
print(f"{'Overall':<15} {overall_total:>8d} {'':>12} {'':>12} {overall_correct/overall_total:>9.2%}")

**Evaluation results discussion:**

The model achieves an **overall accuracy of 98.80%** on the unseen combined test sequences, with most activities classified perfectly. Key observations:

- **Walking and Jumping** achieve 100% sensitivity - these dynamic activities produce highly distinctive signal patterns that the model recognizes without error.
- **Standing** achieves 100% sensitivity but 98.44% specificity - meaning 1 window from another activity was incorrectly classified as Standing.
- **Still** has the lowest sensitivity at 94.12% - one Still window near the transition boundary was misclassified, likely as Standing. This is expected because Standing and Still share similar low-energy, low-variance features, and windows at the boundary contain mixed signals from both activities.

The confusion matrix above shows exactly where the misclassification occurs, confirming that the Standing/Still boundary is the model's weakest point.

Below, we visualize the full decoded test sequences to see exactly where the predictions deviate from ground truth.

In [ ]:
# Visualize decoded combined test sequences
fig, axes = plt.subplots(2, 1, figsize=(14, 8))
fig.suptitle("Viterbi Decoded Combined Sequences (Unseen Test Data)", fontsize=13)

for i in range(2):
    seq = combined_sequences[i]
    true_labels = combined_labels[i]
    pred_path = model.predict(seq)
    boundaries = combined_boundaries[i]
    
    ax = axes[i]
    t = np.arange(len(seq))
    
    # Plot true and predicted state sequences
    ax.step(t, true_labels, where='mid', label='True', color='black', linewidth=2.5, alpha=0.4)
    ax.step(t, pred_path, where='mid', label='Predicted', color='#E91E63', linewidth=1.8)
    
    # Mark activity boundaries with vertical lines
    for b in boundaries[1:-1]:
        ax.axvline(x=b - 0.5, color='gray', linestyle='--', alpha=0.6, linewidth=1)
    
    # Add activity labels at segment centers
    for j in range(len(boundaries) - 1):
        center = (boundaries[j] + boundaries[j+1]) / 2
        seg_label = true_labels[boundaries[j]]
        ax.text(center, 3.4, ACTIVITIES[seg_label], ha='center', fontsize=8, fontstyle='italic')
    
    ax.set_yticks(range(4))
    ax.set_yticklabels(ACTIVITIES, fontsize=9)
    ax.set_ylabel("Activity State")
    ax.set_title(f"Combined Test Sequence {i+1}", fontsize=10)
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(True, alpha=0.2)
    ax.set_ylim(-0.3, 3.8)

axes[-1].set_xlabel("Window Index")
plt.tight_layout()
plt.show()

**Decoded sequence visualization:** The plots above show the true activity labels (black) and the Viterbi-predicted labels (pink) for both combined test sequences. The dashed vertical lines mark the true activity boundaries.

- **Combined Sequence 1** is decoded perfectly - the predicted line matches the true line throughout.
- **Combined Sequence 2** shows a brief misclassification near the Jumping-to-Still transition boundary, where a Still window is classified as Standing. This happens because the boundary window contains mixed sensor data from two activities, and the HMM's self-transition inertia delays the state switch.

This is a realistic result that demonstrates both the strengths and limitations of the HMM approach.

## 12. Analysis and Reflection

### Which activities were easiest/hardest to distinguish?

- **Easiest:** Jumping and Walking. Jumping produces very high accelerometer variance and spectral energy, making it unmistakable. Walking has a characteristic ~2 Hz dominant frequency from the regular step pattern. Both are clearly separable from static activities in the feature space.

- **Hardest:** Standing vs Still. Both are low-movement activities with overlapping feature distributions. The main difference is subtle body sway when standing versus near-zero signal when the phone is placed on a flat surface. This creates ambiguity especially at transition boundaries where windows contain mixed signals.

### How transition probabilities reflect realistic behavior

The learned transition matrix shows high self-transition probabilities (close to 1.0 on the diagonal), which realistically reflects that people tend to continue their current activity rather than rapidly switching every second. This self-transition inertia also acts as a smoothing mechanism, helping the model resist brief spurious state changes caused by noisy sensor observations.

### Effect of sensor noise and sampling rate

Both phones used the same 20ms (50 Hz) sampling rate, simplifying preprocessing since no resampling was needed. However, the two devices (STK-L22 and Infinix X6855) have different sensor hardware, leading to slightly different noise floors and calibration offsets. Z-score normalization helped mitigate these inter-device differences. The 50 Hz rate provides adequate resolution for human activities, which occur at 0.5-5 Hz.

### Potential improvements

1. **More data** from additional participants would improve cross-person generalization
2. **Additional features** such as jerk (derivative of acceleration) and tilt angle could better distinguish Standing from Still
3. **Magnetometer data** could help with orientation-dependent activities
4. **Continuous multi-activity recordings** would provide more realistic transition patterns for training
5. **GMM-HMM** (Gaussian Mixture emissions per state) could capture more complex, multimodal feature distributions